## PRAMAY WANKHADE RBT23CB013

In [1]:
import re, urllib.request
from collections import Counter

In [2]:
# 1. Dictionary with word frequencies (corpus)
text = urllib.request.urlopen("https://norvig.com/big.txt").read().decode().lower()
WORDS = Counter(re.findall(r"[a-z]+", text))

In [3]:
# 2. Levenshtein (edit) distance using dynamic programming
def edit_distance(a, b):
    dp = [[0]*(len(b)+1) for _ in range(len(a)+1)]

    for i in range(len(a)+1): dp[i][0] = i

    for j in range(len(b)+1): dp[0][j] = j

    for i in range(1, len(a)+1):
        for j in range(1, len(b)+1):
            cost = 0 if a[i-1] == b[j-1] else 1
            dp[i][j] = min(dp[i-1][j] + 1,        # deletion
                           dp[i][j-1] + 1,        # insertion
                           dp[i-1][j-1] + cost)   # substitution
    return dp[-1][-1]

In [4]:
# 3. Detection: word not in dictionary => misspelled
def is_misspelled(w):
    return w.lower() not in WORDS


In [8]:
# 4. Suggestion: closest dictionary words, ties broken by frequency
def suggest(w, n=3, max_dist=2):
    w = w.lower()
    cands = [(edit_distance(w, d), -WORDS[d], d)
             for d in WORDS if abs(len(d) - len(w)) <= max_dist]
    cands = [c for c in cands if c[0] <= max_dist]
    return [d for _, _, d in sorted(cands)[:n]]


In [9]:
# 5. Run
test_words = ["recieve", "computar", "definatel", "seperate",
              "occured", "wierd", "adress", "programing", "hello"]


In [10]:
print(f"{'Word':<15}{'Status':<12}Suggestions")
print("-"*50)
for w in test_words:
    if is_misspelled(w):
        print(f"{w:<12}{'MISSPELLED':<12}{suggest(w)}")
    else:
        print(f"{w:<12}{'CORRECT':<12}-")

Word           Status      Suggestions
--------------------------------------------------
recieve     MISSPELLED  ['relieve', 'believe', 'receive']
computar    MISSPELLED  ['computer', 'computers', 'compute']
definatel   MISSPELLED  ['definite', 'definitely']
seperate    MISSPELLED  ['separate', 'separated', 'desperate']
occured     MISSPELLED  ['occurred', 'occur', 'occurs']
wierd       MISSPELLED  ['wiberd', 'wield', 'were']
adress      MISSPELLED  ['dress', 'address', 'across']
programing  MISSPELLED  ['programming']
hello       CORRECT     -
